# 02 — Preprocessing

This notebook loads the raw Quora Question Pairs dataset, produces normalized text columns, and saves a preprocessed dataset to `data/processed/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import nltk

%load_ext autoreload
%autoreload 2

def _find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src").exists() and (p / "data").exists():
            return p
    if (start / "src").exists():
        return start
    return start.parent

PROJECT_ROOT = _find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

from text_preprocessing import (
    normalize_text,
    preprocess_classic_ml,
)

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dmity\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Dmity\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Dmity\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

zip_files = list(RAW_DIR.glob("*.zip"))
csv_files = list(RAW_DIR.glob("*.csv"))

if csv_files:
    csv_path = csv_files[0]
    df = pd.read_csv(csv_path)
elif zip_files:
    zip_path = zip_files[0]
    df = pd.read_csv(zip_path)
else:
    raise FileNotFoundError(f"No .csv or .zip files found in {RAW_DIR}")

df.head()


,id,qid1,qid2,question1,question2,is_duplicate
0,305985,429434,429435,Why is beef banned in India and not pork as well?,Is beef banned in india?,0
1,5193,10230,10231,At what valuation did Homejoy raise money in D...,Should a wealthy founder self-fund his second ...,0
2,123326,199422,199423,How do we judge?,How do I judge my love?,0
3,368557,327674,498931,Are Adderall and meth the same?,Are concerta and meth test the same?,0
4,369226,499645,499646,If you had internet access to only one site fo...,Why is there .co.uk for British internet sites...,0


In [4]:
df["question1"] = df["question1"].fillna("")
df["question2"] = df["question2"].fillna("")

df["q1_norm"] = df["question1"].map(lambda x: normalize_text(x, lowercase=True))
df["q2_norm"] = df["question2"].map(lambda x: normalize_text(x, lowercase=True))

df["q1_classic"] = df["question1"].map(lambda x: preprocess_classic_ml(x, remove_stop_words=True))
df["q2_classic"] = df["question2"].map(lambda x: preprocess_classic_ml(x, remove_stop_words=True))

df[["question1", "q1_norm", "q1_classic", "question2", "q2_norm", "q2_classic"]].head()

,question1,q1_norm,q1_classic,question2,q2_norm,q2_classic
0,Why is beef banned in India and not pork as well?,why is beef banned in india and not pork as well,beef banned india pork well,Is beef banned in india?,is beef banned in india,beef banned india
1,At what valuation did Homejoy raise money in D...,at what valuation did homejoy raise money in d...,valuation homejoy raise money december 2013,Should a wealthy founder self-fund his second ...,should a wealthy founder self fund his second ...,wealthy founder self fund second startup raise...
2,How do we judge?,how do we judge,judge,How do I judge my love?,how do i judge my love,judge love
3,Are Adderall and meth the same?,are adderall and meth the same,adderall meth,Are concerta and meth test the same?,are concerta and meth test the same,concerta meth test
4,If you had internet access to only one site fo...,if you had internet access to only one site fo...,internet access one site rest life site would ...,Why is there .co.uk for British internet sites...,why is there co uk for british internet sites ...,co uk british internet sites fr french ones


In [5]:
keep_cols = [c for c in [
    "id", "qid1", "qid2", "question1", "question2", "is_duplicate",
    "q1_norm", "q2_norm", "q1_classic", "q2_classic"
] if c in df.columns]

out_df = df[keep_cols].copy()

parquet_path = PROCESSED_DIR / "quora_preprocessed.parquet"
csv_path_out = PROCESSED_DIR / "quora_preprocessed.csv"

out_df.to_parquet(parquet_path, index=False)
out_df.to_csv(csv_path_out, index=False)

parquet_path, csv_path_out, out_df.shape

(WindowsPath('D:/Git/Quora-project/data/processed/quora_preprocessed.parquet'),
 WindowsPath('D:/Git/Quora-project/data/processed/quora_preprocessed.csv'),
 (80858, 10))